# Julia - BASE MODEL

In [ ]:
# importing necessary packages

using Images   
using Plots
using Statistics
using FileIO
using ImageIO
using Colors
using LinearAlgebra
using TiffImages
using JLD2
using DelimitedFiles
using SpecialFunctions

## Functions

Four functions were used in this model: "meshgrid", "findnearest". "ModelFromImage", and "diffusion". The purposes of thee functions are listed below:

meshgrid: function to find the index of the nearest value in a vector to assign a units value \
findnearest: function to take difference to find which pixels correspond to which grid cell \
ModelFromImage: function to create a model from the image imported \
diffusion: function to calculate contrubution of diffusion to heat throughout model 

In [ ]:
# function to find the index of the nearest value in a vector to assign a units value************************

function meshgrid(x, y) 
    X = repeat(x', length(y), 1)          # transpose x to make it a row vector
    Y = repeat(y, 1, length(x))           # repeat y to make it a column vector
    return X, Y
end

# x and y: 1D vectors of x and y coords resp
# X and Y: 2D matrices of x and y coords resp

In [ ]:
# findnearest: function to take difference to find which pixels correspond to which grid cell*****************

function findnearest(vec,val) 
    _, idx = findmin(abs.(vec.-val))      # take difference to find closest index
    return idx
end

# vec: 1D vector of values
# val: value to find the nearest index for
# idx: index of nearest value in vec

In [ ]:
# function to create a model from the image imported**********************************************************

function ModelFromImage(file, W, Nx)

    img = load(file)                                                # load the image file
    p,q = size(img)                                                 # get image dimensions

    # setting colours to identify rock types and assign units
    yellow_rgb = [1, 230/255, 128/255]                              # yellow pixel rgb values 
    orange_rgb = [1, 127/255, 39/255]                               # orange pixel rgb values
    red_rgb    = [237/255, 28/255, 36/255]                          # red pixel rgb values
    white_rgb  = [1, 1, 1]                                          # white pixel rgb values

    img_units  = zeros(Int,p,q)                                     # 2D matrix made to be same size as image
    

    # for loop going through every pixel in image
    for i in 1:p, j in 1:q                                         

        pixel = img[i, j]                                           # get pixel color at (i,j)
        rgb = pixel.color                                           # extract RGB values from pixel color

        # extract red, green, and blue components from pixel colour
        r = red(rgb)                                                # red function
        g = green(rgb)                                              # green function
        b = blue(rgb)                                               # blue function
        
        pixel_vec = [r,g,b]                                         # create vector of RGB values

        # check if pixel color is close to one of the defined colors
        if norm(pixel_vec .- yellow_rgb) < 0.01                     # assess each colour one by one
            img_units[i, j] = 1                                     # yellow unit
        elseif norm(pixel_vec .- orange_rgb) < 0.01
            img_units[i, j] = 2                                     # orange unit
        elseif norm(pixel_vec .- red_rgb) < 0.01
            img_units[i, j] = 3                                     # red unit
        else
            img_units[i, j] = 4                                     # white/ outliers unit
        end 
    end

    D   = W*p/q                                                     # calculate depth based on image height and width proportion
    Nz  = floor(Int,Nx*p/q)                                         # calculate number of grid points in z based on image, and take floor to ensure integer value

    # ensure all variables are integers
    q   = Int(q)
    p   = Int(p)
    Nx  = Int(Nx)
    Nz  = Int(Nz)

    # image coordinates for creating grid of image
    hi       = W / q                                               # image grid spacing
    xci      = range(hi/2, stop = W - hi/2, length = q)            # x coordinates for image grid
    zci      = range(hi/2, stop = D - hi/2, length = p)            # same for z
    Xci, Zci = meshgrid(xci, zci)                                  # create meshgrid for image coordinates

    # model or "target" coordinates for creating grid of model
    h     = W/Nx                                                   # model grid spacing
    xc    = range(h/2, stop = W - h/2; length = Nx)                # x coordinates for model grid
    zc    = range(h/2, stop = D - h/2; length = Nz)                # same for z
    Xc,Zc = meshgrid(xc,zc)                                        # create meshgrid for model coordinates

    
    # create empty array for interpolated image units
    img_inter = Array{Int}(undef, Nz, Nx)                          
    

    # for loop going through each grid cell
    for i in 1:Nz
        for j in 1:Nx                                              
            
            x = xc[j]                                              # get x coordinate of pixel
            z = zc[i]                                              # same for z coords
            p_j = findnearest(xci, x)                              # find nearest x coordinate in image grid
            p_i = findnearest(zci, z)                              # same for z coords

            img_inter[i, j] = img_units[p_i, p_j]                  # assign interpolated image unit to model grid
        end
    end


    units = Int.(img_inter)                                        # convert interpolated image units to integers
    return units, Nz, D
end

# file:   path to image file
# W:      width of model in metres
# Nx:     number of grid points in x direction
# units:  matrix of rock unit numbers
# D:      depth of model i metres
# Nz:     number of grid poits in z direction

In [ ]:
# function to calculate contrubution of diffusion to heat throughout model*****************************************

function diffusion(f, dTdz_0, KT, dx, ix, iz, Ttop)
    
    kx = (KT[:, ix[1:end-1]] .+ KT[:, ix[2:end]]) ./ 2                    # average conductivity in x for cell faces using ghost indexes
    kz = (KT[iz[1:end-1], :] .+ KT[iz[2:end], :]) ./ 2                    # same thing but for z now
    
    Tx = f[:, vec(ix)]                                                    # split ghost matrix of temperature in x
    Tz = f[vec(iz),:]                                                     # same thing but z now
   
    Tx[:,1]   .= Tx[:,2]                                                  # set insulated temperature at left boundary
    Tx[:,end] .= Tx[:,end-1]                                              # set insulated temperature at right boundary

    Tz[1,:]   .= Ttop                                                     # set isothermal temperature at top boundary
    Tz[end,:] .= Tz[end-1,:]                                              # set insulated temperature at bottom boundary


    qx = -kx .* diff(Tx, dims = 2) ./ dx                                  # horizontal heat flux using x variables
    qz = -kz .* diff(Tz, dims = 1) ./ dx                                  # same thing but z now
    
    qz[end, :] .= -kz[end, :] .* dTdz_0[2]                                # set constant geothermal flux at bottom boundary

   
    dTdt = -((diff(qx, dims = 2) ./ dx) .+ (diff(qz, dims = 1) ./ dx))    # calculate flux balance for rate of change

    return dTdt
end

# f:       2D matrix of temperature values 
# dTdz_0:  geothermal gradient at bottom boundary
# KT:      2D matrix of thermal diffusivity
# dx:      grid spacing
# ix, iz:  ghost indexes for x and y direction respectively
# Ttop:    temperature at top boundary

## Set Parameters

Parameters set for grid as well as matrix created to call on material properties. Finally numerical parameters set.

In [ ]:
# grid constants ***************************************************************************************

W    = 4e3; # width of the domain, should match image  [m]
dx   = 20; # spacing between each grid point - resolution  [m]
Nx   = Int64(W/dx); # number of column in x direction ie 100  
file = "//campus.gla.ac.uk/isi/stud-file/Documents/Julia/image.tiff"
units, Nz, D = ModelFromImage(file, W, Nx) # set outputs of function as variables


# define a matrix of material properties for each unit
matprop = [
    1    2.6     850     1100    0.8244e-6    1e-21;   # host lewisian rock
    2    2.6     850     1100    0.8244e-6    1e-14;   # damage zone
    3    2.6     850     1100    0.8244e-6    1e-17;   # fault core
    4    1e-6    1       1000    0.0              0;   # air/water
]

# setting material properties for each unit in the matrix
sigma = matprop[units, 2]
rho0  = matprop[units, 3]
Cp    = matprop[units, 4]
Hr    = matprop[units, 5]
k_p   = matprop[units, 6]

# sigma: Themal conductivity [W/(mK)]
# Cp: Specfic heat capacity  [J/(kgK)]
# rho: Density               [kg/m^3]
# Hr: Heat production rate   [W/m^3]
# ko: Diffusivity constant   [m^2/s]
# k_p: Permeability          [m^2]

dTdz_0    = [0, 35/1000]                              # geothermal gradient at bottom boundary [°C/m]
Ttop      = 8.2                                       # initial temperature [°C]
Tair      = 8.2                                       # air temperature [°C]
nop       = 1000                                      # number of time steps to print at
yr        = 3600*24*365                               # seconds in a year [s]
tend      = 1e6 * yr                                  # end time of simulation [s]
CFL       = 0.8                                       # Courant-Friedrichs-Lewy condition for stability
mu        = 1e-3                                      # dynamic viscosity [Pa.s]
g         = 9.81                                      # gravitational acceleration [m/s^2]
aT        = 2.5e-5                                    # thermal expansion coefficient [1/°C]
rho_fluid = 1025                                      # reference density [kg/m^3]
alpha     = 0.9                                       # numerical scaling factor
beta      = 0.9                                       # numerical scaling factor
gamma     = 0.25                                      # numerical scaling factor
tol       = 1e-8                                      # tolerance for residual
psi       = 0.0001                                    # permeability decay constant

## Model Setup

Create numerical grid, index vectors, and set initial conditions.

In [ ]:
# grid setup and cell coordinates in a list*****************************************************************************

x_cc = range(dx/2, stop = W - dx/2; length = Nx)             # cell center coordinates in x direction
z_cc = range(dx/2, stop = D - dx/2; length = Nz)             # same in z coords
x_fc = range(0, stop = W; length = Nx+1)                     # face center coordinates in x direction
z_fc = range(0, stop = D; length = Nz+1)                     # same in z coords


# set up meshgrid 
Xc, Zc = meshgrid(x_cc,z_cc)                                 # create meshgrid for cell centers

ix  = reshape(vcat(1, collect(1:Nx), Nx), 1, :)              # create ghost indexes for x direction, (1 x Nx+2)
iz  = reshape(vcat(1, collect(1:Nz), Nz), :, 1)              # same for z direction, (Nz+2 x 1)
ix5 = reshape(vcat(1,1, collect(1:Nx), Nx, Nx), 1, :)        # create ghost indexes for x direction, (1 x Nx+2)
iz5 = reshape(vcat(1,1, collect(1:Nz), Nz, Nz), :, 1)        # same for z direction, (Nz+2 x 1)


# set initial temperature conditions
T = Ttop .+ dTdz_0[2] .* Zc                                  # initial temperature distribution gradient based on geothermal gradient
is_air = units .== 4                                         # set air units
is_rock = units .!= 4                                        # set rock units
T[is_air] .= Ttop                                            # set temperature in air

KT = sigma ./ rho0 ./ Cp                                     # diffusivity constant calculated
dt = CFL * (dx/2)^2 / maximum(KT)                            # time step based on CFL condition and diffusivity


# advection setup - not used in this model ****************************************************************************************************

exponent  = - psi * Zc                                       # exponential decrease in permeability
KD        = (k_p .* exp.(exponent) )/mu                      # full darcy mobility
rho       = rho0 .* (1 .- aT .* (T .- Tair))                 # initial density based on thermal expansion
dtau = (dx/2)^2 ./ KD                                        # advection iteration step

u     = zeros(Nz, Nx)                                        # initial horizontal velocity
w     = zeros(Nz, Nx)                                        # initial vertical velocity                                
P     = zeros(Nz, Nx)                                        # initial pressure
dTdt  = zeros(Nz, Nx)                                        # rate of change of temperature
upd_P = zeros(Nz, Nx)  

for j in 1:Nx
    for i in 1:Nz
        if is_rock[i, j]                                     # only for cells that are rocks
            
            depth = z_cc[i]                                  # get depth from z_cc
            P[i, j] = rho_fluid * g * depth                  # calculate and assign pressure

        end
    end
end



## Main numerical loop

Use a residual method to calculate diffusion and heat production along a time series. 

In [ ]:
# main operation to combine all functions and run the model*******************************************************


t = 0                                                                                # initial time
tau = 0                                                                              # time step counter

# set up for analytical solution
Tinf = dTdz_0[2] * D                                                                 
DeltaT = 20

all_residuals = Vector{Vector{Float64}}()                                            # initialise residuals storage


# loop through time by timesteps until time greater than or equal to end time
while t <= tend                                                                      
    
    t += dt                                                                          # add time step to time each iteration
    tau += 1                                                                         # increment time step counter
    
    T[units .== 4] .= Ttop                                                           # set air temperature to 10C for air/water unit

    # store old values for residual calculation
    To = copy(T)                                                                     
    dTdto = copy(dTdt)        

    # intialise normalised residual 
    resnorm = 1                                                                      
    res_history = Float64[]


    # loop through calculating solution until residual is less than or equal to tolerance
    while resnorm  >= tol

        dTdt = diffusion(T, dTdz_0, KT, dx, ix, iz, Ttop)  + Hr ./rho0 ./Cp          # heat transport calculation                          
    
        res_T = (T-To)/dt - (dTdt + dTdto)/2                                         # residual calculation
        upd_T = - alpha * res_T * dt                                                 # update to temperature
        T    .= T + upd_T

        resnorm = norm(upd_T[:]) / (norm(T[:]) + eps())                              # normalise residual
        push!(res_history, resnorm)                                                  # store convergence at each iteration
    end

    
    push!(all_residuals, copy(res_history))                                          # store final residual from each time step
        
    @assert !any(isnan.(T)) "NaNs introduced in T at time step $tau, time = $t"
    
    # calculate error and print results
    if tau % nop == 0 || t >= tend                                                   
        Tana = Tinf .+ DeltaT .* erfc.(z_cc ./ (2*sqrt.(KD .* t)))                   # calculate analytical solution at each time step
        Err = norm(T - Tana, 2) / norm(Tana, 2)                                      # calculate normalised error

        avgT   = mean(T)
        minT   = minimum(T)
        maxT   = maximum(T)
        deltaT = maximum(abs.(T .- To))

        println("Step: $tau | Time: $(round(t, digits=2)) s | ",                     # print statements to monitor solutions
            "Avg T: $(round(avgT, digits=2))°C | ",
            "Min: $(round(minT, digits=2))°C | ",
            "Max: $(round(maxT, digits=2))°C | ",
            "ΔTmax: $(round(deltaT, digits=4))°C |",
            "Error:$(Err)°C")

    end

end
print("Simulation finished")

## Time Convergence

Plot time convergence to proe residual setup works.

In [ ]:
# extract residuals from relevant time steps

res300000 = all_residuals[300000]
res100000 = all_residuals[100000]
res1000 = all_residuals[1000]
res100 = all_residuals[100]
res10 = all_residuals[10]

xticks = 1:length(res10)


plot(res10, yscale = :log10, xlabel="Iteration", ylabel="Resnorm (log10)", xticks = 1:13, title="Nonlinear convergence", color = :teal, legend = true, label = "Time step: 10")

plot!(res100, yscale = :log10, color = :slateblue3, label = "Time step: 100")
plot!(res1000, yscale = :log10, color = :fuchsia, label = "Time step: 1000")
plot!(res100000, yscale = :log10, color = :red, label = "Time step: 100000")
plot!(res300000, yscale = :log10, color = :yellow2, label = "Time step: 300000", linestyle = :dash)

savefig("Conv-time-5.pdf")

## Plotting Temperature

In [ ]:
# make plot of temperature with isotherm contours

gr() 

p_final = plot(
    x_cc, z_cc, T;                       # x, z, and colour values 
    st = :heatmap,                       # plot type heatmap
    aspect_ratio = 1,
    xlabel = "x [m]",                    # plot labels
    ylabel = "z [m]",
    title = "Base Parameters",
    colorbar_title = "Temperature [°C]",
    c = :thermal,
    yflip = true,                        # flip y axis so z increases with depth
    xlims = (0, 4000),
    fontfamily = "Times New Roman",
    size = (800, 600))

# create contours to show surface and isotherm positions
contour!(p_final, x_cc, z_cc, T, 
        levels=[8.3, 25, 40, 90, 150], 
        linecolor=[:white, :red, :red, :red, :red], 
        linewidth=[1, 2, 2, 2], 
        label = false, colorbar=true) 

# label the isotherms on the plot        
annotate!(p_final, [
    (2000, 4095, text("150°C", 6, :red)),
    (2000, 2420, text("90°C", 6, :red)),
    (2000, 1030, text("40°C", 6, :red)),
    (2000, 600, text("25°C", 6, :red))])


savefig(p_final, "Temperature_Diff_decr_Cp-5.pdf")
p_final


## Save Ouput Data

Save data as JLD2 files and CSV file

In [ ]:
# save data as JLD2 files and CSV file

# JLD2 file
filename = "sim_results_Diff_decr_Cp-5.jld2"
jldsave(filename; T_final=T, time_years=t/yr, Nx, Nz, dx, D, W, u_final=u, w_final=w, P_final=P, units)
println("Final simulation data saved to: $filename")

data_T_sim5 = load("sim_results_Diff_decr_Cp-5.jld2")
T_sim5 = data_T_sim5["T_final"]
writedlm("T_sim5.csv", T_sim5, ',')

## Opening files

In [ ]:
# load data to check file

filename_to_load = "sim_results_Diff_decr_Cp-5.jld2"

# Open the file and load the data
data_file = jldopen(filename_to_load, "r")
loaded_T = read(data_file, "T_final")
loaded_time_years = read(data_file, "time_years")
loaded_Nx = read(data_file, "Nx")
loaded_Nz = read(data_file, "Nz")

# You can now work with the loaded variables
println("Data loaded successfully for Nx = $loaded_Nx")
println("Final temperature matrix has dimensions: $(mean(loaded_T))")
println("Final simulation time: $(loaded_time_years) years")

close(data_file)